# Surface Routing Backends

Two backends implement `SurfaceRouter`. Each makes different algorithmic
choices, with different data requirements and tradeoffs.

| Backend | Algorithm | Data | Best for |
|---|---|---|---|
| `OsmnxSurfaceRouter` | Dijkstra on OSM street graph | OSM (Overpass) | Urban, road-network fidelity |
| `FrictionSurfaceRouter` | MCP_Geometric on friction raster | Friction GeoTIFF or speed override | Off-road walking, global coverage |

Both implement the same interface — `compute_surface(dest, bbox, res, mode)` —
so swapping is one line.

**What's covered**
1. `OsmnxSurfaceRouter` — network-aware urban routing
2. `FrictionSurfaceRouter` — raster least-cost path
3. Visual comparison of the two surfaces
4. `SurfaceRouter` protocol — writing a custom backend
5. When to use each backend


## Setup

In [ ]:
from idd_travel_time.access import (
    BoundingBox, OsmnxSurfaceRouter, FrictionSurfaceRouter,
)
from idd_travel_time.constants import TransportMode
import numpy as np, matplotlib.pyplot as plt

DESTINATION = (-1.3006, 36.8073)   # KNH, Nairobi
BBOX = BoundingBox(west=36.76, south=-1.36, east=36.86, north=-1.24)
RESOLUTION = 0.01
MODE = TransportMode.WALKING


## 1 · OsmnxSurfaceRouter — network-aware routing

Downloads the OSM street graph, runs single-source Dijkstra from the
destination node, then assigns travel times to grid cells via KD-tree
nearest-node lookup.

**Strengths:** follows actual roads; correct turn costs; one-way streets.
**Limitations:** slower first run (Overpass download); needs a street
network (doesn't model off-road walking across fields).


In [ ]:
osmnx_router = OsmnxSurfaceRouter()
print("Computing OsmnxSurfaceRouter surface …")
osmnx_surface = osmnx_router.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)
print(osmnx_surface)
print(osmnx_surface.summary_stats())


## 2 · FrictionSurfaceRouter — raster least-cost path

Uses `skimage.graph.MCP_Geometric` on a friction raster (minutes/metre
per cell). Works off-road; naturally multi-source; global coverage with
MAP friction data.

For this demo we use a *uniform speed override* (5 km/h walking) — which
treats every cell identically. In production you'd supply a real friction
GeoTIFF from MAP or a custom raster built from OSM + land cover + DEM.


In [ ]:
friction_router = FrictionSurfaceRouter(speed_kph_override=5.0)
print("Computing FrictionSurfaceRouter surface (uniform 5 kph) …")
friction_surface = friction_router.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)
print(friction_surface)


## 3 · Visual comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

extent = [BBOX.west, BBOX.east, BBOX.south, BBOX.north]
kw = dict(cmap="RdYlGn_r", vmin=0, vmax=60, origin="upper",
          aspect="auto", extent=extent)

# OsmnxSurfaceRouter
ax = axes[0]
im = ax.imshow(osmnx_surface.data, **kw)
ax.scatter([DESTINATION[1]], [DESTINATION[0]], c="blue", s=120, zorder=5)
ax.set_title("OsmnxSurfaceRouter\n(Dijkstra on road network)")
plt.colorbar(im, ax=ax, label="min")

# FrictionSurfaceRouter
ax = axes[1]
ax.imshow(friction_surface.data, **kw)
ax.scatter([DESTINATION[1]], [DESTINATION[0]], c="blue", s=120, zorder=5)
ax.set_title("FrictionSurfaceRouter\n(MCP on uniform 5 kph raster)")

# Difference
ax = axes[2]
from idd_travel_time.access import compare_surfaces
comp = compare_surfaces(osmnx_surface, friction_surface)

diff_data = (osmnx_surface.resample(RESOLUTION).data -
             friction_surface.resample(RESOLUTION).data)
im2 = ax.imshow(diff_data, cmap="RdBu", vmin=-20, vmax=20,
                origin="upper", aspect="auto", extent=extent)
ax.set_title(f"Difference (osmnx − friction)\nRMSD={comp['rmsd']:.1f} min")
plt.colorbar(im2, ax=ax, label="min")

plt.tight_layout()
plt.show()

print(f"Pixel-wise comparison:")
for k, v in comp.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")


## 4 · Writing a custom backend

Any class implementing `SurfaceRouter` is a valid backend — even one that
wraps a REST API, a pre-computed lookup table, or an R model.


In [ ]:
from idd_travel_time.access import SurfaceRouter, TravelTimeSurface, BoundingBox
from idd_travel_time.access.backends.osmnx_surface import _make_grid
from idd_travel_time.constants import TransportMode
import numpy as np

class EuclideanRouter(SurfaceRouter):
    """Straight-line (crow-flies) distance at constant speed.
    Not realistic but useful as a lower-bound benchmark."""

    def __init__(self, speed_kph: float = 5.0) -> None:
        self._speed = speed_kph

    @property
    def name(self) -> str:
        return f"euclidean_{self._speed}kph"

    def compute_surface(self, destination, bbox, resolution_deg, mode, *,
                        transit_profile=None):
        dest_lat, dest_lon = destination
        lats, lons = _make_grid(bbox, resolution_deg)

        # Approximate great-circle distance in km
        dlat = (lats - dest_lat) * 111.0
        dlon = (lons - dest_lon) * 111.0 * np.cos(np.radians(dest_lat))
        dist_km = np.sqrt(dlat**2 + dlon**2)

        time_min = (dist_km / self._speed * 60).astype(np.float32)

        return TravelTimeSurface.from_arrays(
            data=time_min,
            west=bbox.west, north=bbox.north,
            resolution_deg=resolution_deg,
            backend=self.name,
            destination=destination,
        )

# Use it just like any built-in backend
euc = EuclideanRouter(speed_kph=5.0)
euc_surface = euc.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)
print(euc_surface)
print(f"Euclidean surface mean: {euc_surface.summary_stats()['mean']:.1f} min")


## 5 · When to use each backend

| Scenario | Recommended backend |
|---|---|
| Urban area, walking/cycling | `OsmnxSurfaceRouter` |
| Rural area with few roads | `FrictionSurfaceRouter` (MAP friction) |
| Global or continental coverage | `FrictionSurfaceRouter` (MAP 1 km) |
| Sub-Saharan Africa at 100 m | `FrictionSurfaceRouter` (Hierink 100 m) |
| Off-road children walking (Uganda/Tanzania) | `FrictionSurfaceRouter` (Watmough 20 m) |
| Formal transit (Helsinki, São Paulo) | r5py backend (future) |
| Informal transit (Lagos, Nairobi) | `OsmnxSurfaceRouter` + `TransitProfile` |
| Lower-bound reference | `EuclideanRouter` (custom, above) |
